# Java Code Refinement - Data Processing
This notebook prepares the CodeXGlue Java refinement dataset for training our bug detection and fixing model

## 1. Setup Environment

In [ ]:
# Install requirements
!pip install -q transformers==4.40.0 datasets==2.19.0 tree-sitter tree-sitter-java

# Clone project repository
!git clone -q https://github.com/yourusername/BugFixer.git
%cd BugFixer

# Install project dependencies
!pip install -q -r requirements.txt

# Build tree-sitter languages
from tree_sitter import Language
Language.build_library(
    'build/my-languages.so',
    ['tree-sitter-java']
)

## 2. Load and Inspect Dataset

In [ ]:
from datasets import load_dataset

# Load CodeXGlue Java refinement dataset
dataset = load_dataset("code_x_glue_cc_code_refinement", "java")

# Show dataset structure
print(f"Dataset features: {dataset['train'].features}")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Show a sample
sample = dataset['train'][0]
print("\nSample buggy code:")
print(sample['buggy'])
print("\nSample fixed code:")
print(sample['fixed'])

## 3. Preprocessing Pipeline

In [ ]:
import sys
sys.path.append('/kaggle/working/BugFixer')

from src.data_processing import JavaPreprocessor, BugTypeClassifier

# Initialize processors
preprocessor = JavaPreprocessor()
classifier = BugTypeClassifier()

def clean_code(example):
    """Clean buggy and fixed code"""
    try:
        example['buggy_clean'] = preprocessor.clean_code(example['buggy'])
        example['fixed_clean'] = preprocessor.clean_code(example['fixed'])
        return example
    except Exception as e:
        print(f"Error cleaning code: {e}")
        return None

def classify_error(example):
    """Classify error type"""
    try:
        example['error_type'] = classifier.classify(
            example['buggy_clean'], 
            example['fixed_clean']
        )
        return example
    except Exception as e:
        print(f"Error classifying error: {e}")
        return None

# Apply preprocessing
print("Cleaning code...")
dataset = dataset.map(clean_code, batched=False)

# Apply error classification
print("Classifying error types...")
dataset = dataset.map(classify_error, batched=False)

# Filter failed examples
dataset = dataset.filter(lambda x: x is not None)

## 4. Tokenization

In [ ]:
from transformers import AutoTokenizer

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base-java")
MAX_LENGTH = 256

def tokenize_example(example):
    """Tokenize buggy and fixed code"""
    try:
        # Tokenize inputs (buggy code)
        inputs = tokenizer(
            example['buggy_clean'],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Tokenize targets (fixed code)
        targets = tokenizer(
            example['fixed_clean'],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": inputs["input_ids"][0].numpy().tolist(),
            "attention_mask": inputs["attention_mask"][0].numpy().tolist(),
            "labels": targets["input_ids"][0].numpy().tolist(),
            "error_label": classifier.ERROR_TYPES.index(example['error_type'])
        }
    except Exception as e:
        print(f"Tokenization error: {e}")
        return None

# Apply tokenization
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize_example, batched=False)

# Filter failed tokenizations
tokenized_dataset = tokenized_dataset.filter(lambda x: x is not None)

## 5. Dataset Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Error type distribution
error_counts = {}
for split in ['train', 'validation', 'test']:
    error_types = [classifier.ERROR_TYPES[label] for label in tokenized_dataset[split]['error_label']]
    error_counts[split] = pd.Series(error_types).value_counts()

# Plot distribution
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for i, split in enumerate(['train', 'validation', 'test']):
    error_counts[split].plot.pie(
        ax=ax[i], 
        autopct='%1.1f%%',
        title=f'{split.capitalize()} Split Error Distribution'
    )
plt.tight_layout()
plt.savefig('error_distribution.png')
plt.show()

# Token length analysis
def token_lengths(input_ids):
    return sum(1 for token_id in input_ids if token_id != tokenizer.pad_token_id)

train_lengths = [token_lengths(ids) for ids in tokenized_dataset['train']['input_ids']]

plt.figure(figsize=(10, 6))
plt.hist(train_lengths, bins=50, alpha=0.7, color='skyblue')
plt.axvline(MAX_LENGTH, color='red', linestyle='dashed', linewidth=1)
plt.title('Token Length Distribution')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.savefig('token_lengths.png')
plt.show()

print(f"Samples exceeding max length: {sum(1 for l in train_lengths if l > MAX_LENGTH)}/{len(train_lengths)}")

## 6. Save Processed Dataset

In [ ]:
import os
import shutil
from datetime import datetime

# Create output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"/kaggle/working/processed_java_refinement_{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Save dataset
tokenized_dataset.save_to_disk(output_dir)

# Save visualizations
shutil.copy('error_distribution.png', output_dir)
shutil.copy('token_lengths.png', output_dir)

# Create dataset description
with open(os.path.join(output_dir, 'README.md'), 'w') as f:
    f.write("# Java Code Refinement Dataset\n\n")
    f.write("Processed dataset for training Java bug detection and fixing models\n\n")
    f.write("## Statistics\n")
    f.write(f"- Train samples: {len(tokenized_dataset['train'])}\n")
    f.write(f"- Validation samples: {len(tokenized_dataset['validation'])}\n")
    f.write(f"- Test samples: {len(tokenized_dataset['test'])}\n\n")
    f.write("## Features\n")
    f.write("- `input_ids`: Token IDs for buggy code\n")
    f.write("- `attention_mask`: Attention mask\n")
    f.write("- `labels`: Token IDs for fixed code\n")
    f.write("- `error_label`: Error type classification (0-3)\n")

# Create dataset ZIP
shutil.make_archive(output_dir, 'zip', output_dir)

print(f"\nProcessing complete! Dataset saved to {output_dir}.zip")

## 7. Create New Dataset Version (Optional)

In [ ]:
# Only run if you want to create a new Kaggle dataset
"""
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# Create new dataset version
api.dataset_create_version(
    folder=output_dir,
    version_notes=f"Processed Java refinement dataset {timestamp}",
    convert_to_csv=False,
    delete_old_versions=False
)

print("New dataset version created!")
"""